In [33]:
/*basic math expression evaluator
input: 3 + 4 * 2 / (1 - 5)
tokenizer: [Number, Op, Number, Op, Number, Op, LParen, Number, Op, Number, RParen]
ir: 3 4 2 * 1 5 - / +
output: 1
*/

enum TokenKind {
    Num = "num",
    Op = "op",
    LParen = "lparen",
    RParen = "rparen",
}

enum Op {
    Add = "add",
    Sub = "sub",
    Mult = "mult",
    Div = "div",
}

// I quit 03-31-2026; nvm apparently i locked tf in

function opFromChar(char: string): Op {
    switch (char) {
        case "+":
            return Op.Add;
        case "-":
            return Op.Sub;
        case "*":
            return Op.Mult;
        case "/":
            return Op.Div;
        default:
            throw new Error("Invalid input");
    }
}

function opPriority(op: Op): number {
    switch (op) {
        case Op.Add:
        case Op.Sub:
            return 10;
        case Op.Mult:
        case Op.Div:
            return 20;
    }
}

function opApply(op: Op, a: number, b: number): number {
    switch (op) {
        case Op.Add:
            return a + b;
        case Op.Sub:
            return a - b;
        case Op.Mult:
            return a * b;
        case Op.Div:
            return a / b;
    }
}

type Token =
    | { kind: TokenKind.Num; value: number }
    | { kind: TokenKind.Op; op: Op }
    | { kind: TokenKind.LParen }
    | { kind: TokenKind.RParen };

type RpnToken =
    | { kind: TokenKind.Num; value: number }
    | { kind: TokenKind.Op; op: Op };

interface ExpressionEvaluator<T, U> {
    tokenize(input: string): T[];
    parse(tokens: T[]): U[];
    evaluate(tokens: U[]): number;
}

class SimpleExpressionEvaluator
    implements ExpressionEvaluator<Token, RpnToken> {
    tokenize(input: string): Token[] {
        let tokens: Token[] = [];
        // let number_stack = [] // TODO
        for (const char of input) {
            if (char >= "0" && char <= "9") {
                tokens.push({ kind: TokenKind.Num, value: char - "0" });
            } else if (
                char == "+" || char == "-" || char == "*" || char == "/"
            ) {
                tokens.push({ kind: TokenKind.Op, op: opFromChar(char) });
            } else if (char == "(") {
                tokens.push({ kind: TokenKind.LParen });
            } else if (char == ")") {
                tokens.push({ kind: TokenKind.RParen });
            }
        }
        return tokens;
    }
    parse(tokens: Token[]): RpnToken[] {
        let output: RpnToken[] = [];
        let stack: Token[] = [];

        for (const token of tokens) {
            switch (token.kind) {
                case TokenKind.Num:
                    output.push(token);
                    break;
                case TokenKind.Op:
                    if (stack.length > 0) {
                        const peek = stack[stack.length - 1];
                        if (peek.kind == TokenKind.Op) {
                            if (
                                opPriority(peek.op) >= opPriority(token.op)
                            ) output.push(stack.pop());
                        }
                    }
                    stack.push(token);
                    break;
                case TokenKind.LParen:
                    stack.push(token);
                    break;
                case TokenKind.RParen:
                    for (
                        let pop = stack.pop();
                        pop?.kind != TokenKind.LParen;
                        pop = stack.pop()
                    ) {
                        output.push(pop);
                    }
                    break;
            }
        }

        for (
            let pop = stack.pop();
            pop != undefined;
            pop = stack.pop()
        ) {
            output.push(pop);
        }

        return output;
    }
    evaluate(tokens: RpnToken[]): number {
        if (tokens.length < 1) {
            throw new Error("Must not be empty");
        }

        let stack: number[] = [];
        for (const token of tokens) {
            switch (token.kind) {
                case (TokenKind.Num):
                    stack.push(token.value);
                    break;
                case (TokenKind.Op):
                    const b = stack.pop();
                    const a = stack.pop();
                    stack.push(opApply(token.op, a, b));
                    break;
            }
        }
        return stack[0];
    }
}

const createCalculator = (
    evaluator: ExpressionEvaluator,
): (string) => number => {
    return (input: string) => {
        const tokens = evaluator.tokenize(input);
        const rpnTokens = evaluator.parse(tokens);
        return evaluator.evaluate(rpnTokens);
    };
};
const calculate = createCalculator(
    new SimpleExpressionEvaluator(),
);
[
    calculate(`(3+5)*(7-2)/4`),
    calculate(`3 + 4 * 2 / (1 - 5)`),
    calculate(`(8+2)*(5-0)/3`),
    calculate(`(9-4)*(6+3)/2`),
];


[ 10, 1, 16.666666666666668, 22.5 ]